In [0]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  NOTEBOOK 00 — CATALOG SETUP + CONFIG                                   ║
# ║  Run ONCE before any other notebook.                                    ║
# ║  Creates: catalog → schemas → all Delta tables                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# MEDALLION ARCHITECTURE
# ──────────────────────────────────────────────────────────────────────────
#
#   BRONZE  (01_bronze)
#     └─ Raw landing zone. All strings. Append-only. Never modified.
#        schema_drift_flag marks rows that arrived with missing columns.
#
#   SILVER  (02_silver)
#     ├─ dim_customer_scd2      → cleaned, SCD2, PII masked
#     ├─ dim_product_scd2       → cleaned, SCD2
#     ├─ fact_sales_clean       → complete rows (price resolved)
#     └─ fact_sales_quarantine  → incomplete rows (no price) → PENDING/RESOLVED
#
#   GOLD    (03_gold)
#     ├─ dim_customer  → current snapshot only, surrogate key
#     ├─ dim_product   → current snapshot only, surrogate key
#     ├─ dim_date      → date spine
#     └─ fact_sales    → star schema, surrogate keys, BI-ready
#
#   METADATA (04_metadata)
#     ├─ processed_files   → CDC state store (which files are done)
#     ├─ pipeline_runs     → execution log for SLA monitoring
#     ├─ schema_versions   → schema change history for audit
#     ├─ dq_results        → data quality check results
#     └─ quarantine_log    → quarantine trend tracking

# ══════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG — imported by all other notebooks
# ══════════════════════════════════════════════════════════════════════════════

CATALOG  = "retail_data_project"
BRONZE   = f"{CATALOG}.01_bronze"
SILVER   = f"{CATALOG}.02_silver"
GOLD     = f"{CATALOG}.03_gold"
METADATA = f"{CATALOG}.04_metadata"

# Volume path — where raw CSV files are uploaded
RAW_BASE = f"/Volumes/{CATALOG}/01_bronze/raw_data"

# Bronze tables
CUSTOMER_TABLE = f"{BRONZE}.customers"
PRODUCT_TABLE  = f"{BRONZE}.products"
SALES_TABLE    = f"{BRONZE}.sales"

# Silver tables
CUSTOMER_SILVER  = f"{SILVER}.dim_customer_scd2"
PRODUCT_SILVER   = f"{SILVER}.dim_product_scd2"
SALES_CLEAN      = f"{SILVER}.fact_sales_clean"
SALES_QUARANTINE = f"{SILVER}.fact_sales_quarantine"

# Gold tables
DIM_CUSTOMER = f"{GOLD}.dim_customer"
DIM_PRODUCT  = f"{GOLD}.dim_product"
DIM_DATE     = f"{GOLD}.dim_date"
FACT_SALES   = f"{GOLD}.fact_sales"

# Metadata tables
META_TABLE      = f"{METADATA}.processed_files"
PIPELINE_RUNS   = f"{METADATA}.pipeline_runs"
SCHEMA_VERSIONS = f"{METADATA}.schema_versions"
DQ_RESULTS      = f"{METADATA}.dq_results"
QUARANTINE_LOG  = f"{METADATA}.quarantine_log"

print("Config loaded ✓")
print(f"  Catalog : {CATALOG}")
print(f"  Raw base: {RAW_BASE}")


# ══════════════════════════════════════════════════════════════════════════════
# DDL — BRONZE TABLES
# All columns are STRING — bronze is a raw landing zone, never typed here.
# schema_drift_flag = True means the row arrived with one or more missing
# columns (filled with NULL). Silver will handle imputation or rejection.
# Partitioned by received_date for efficient time-range queries.
# ══════════════════════════════════════════════════════════════════════════════

DDL_BRONZE_CUSTOMERS = f"""
CREATE TABLE IF NOT EXISTS {CUSTOMER_TABLE} (
    customer_id        STRING    COMMENT 'Source customer identifier (PK in source)',
    name               STRING    COMMENT 'Customer full name',
    email              STRING    COMMENT 'Raw email — PII, masked in silver',
    city               STRING    COMMENT 'Customer city',
    signup_date        STRING    COMMENT 'Raw signup date string — cast in silver',
    received_date      DATE      COMMENT 'Date this row arrived in bronze',
    source_file        STRING    COMMENT 'Full path of the source CSV file',
    ingestion_ts       TIMESTAMP COMMENT 'Exact timestamp when row was ingested',
    schema_drift_flag  BOOLEAN   COMMENT 'True if row arrived with missing columns (null-filled)'
)
USING DELTA
PARTITIONED BY (received_date)
COMMENT 'Bronze customers — raw, append-only, never overwritten or deleted'
"""

DDL_BRONZE_PRODUCTS = f"""
CREATE TABLE IF NOT EXISTS {PRODUCT_TABLE} (
    product_id         STRING    COMMENT 'Source product identifier (PK in source)',
    product_name       STRING    COMMENT 'Product display name',
    category           STRING    COMMENT 'Product category',
    price              STRING    COMMENT 'Raw price string — may be NULL/empty, cast in silver',
    received_date      DATE      COMMENT 'Date this row arrived in bronze',
    source_file        STRING    COMMENT 'Full path of the source CSV file',
    ingestion_ts       TIMESTAMP COMMENT 'Exact timestamp when row was ingested',
    schema_drift_flag  BOOLEAN   COMMENT 'True if row arrived with missing columns (null-filled)'
)
USING DELTA
PARTITIONED BY (received_date)
COMMENT 'Bronze products — raw, append-only, never overwritten or deleted'
"""

DDL_BRONZE_SALES = f"""
CREATE TABLE IF NOT EXISTS {SALES_TABLE} (
    order_id           STRING    COMMENT 'Source order identifier (PK in source)',
    customer_id        STRING    COMMENT 'FK to customers',
    product_id         STRING    COMMENT 'FK to products',
    quantity           STRING    COMMENT 'Raw quantity string — cast in silver',
    order_date         STRING    COMMENT 'Raw order date string — cast in silver',
    received_date      DATE      COMMENT 'Date this row arrived in bronze',
    source_file        STRING    COMMENT 'Full path of the source CSV file',
    ingestion_ts       TIMESTAMP COMMENT 'Exact timestamp when row was ingested',
    schema_drift_flag  BOOLEAN   COMMENT 'True if row arrived with missing columns (null-filled)'
)
USING DELTA
PARTITIONED BY (received_date)
COMMENT 'Bronze sales — raw, append-only, never overwritten or deleted'
"""


# ══════════════════════════════════════════════════════════════════════════════
# DDL — SILVER TABLES
# ══════════════════════════════════════════════════════════════════════════════

DDL_SILVER_DIM_CUSTOMER_SCD2 = f"""
CREATE TABLE IF NOT EXISTS {CUSTOMER_SILVER} (
    customer_id    STRING    NOT NULL COMMENT 'Business key from source',
    name           STRING    COMMENT 'Cleaned customer full name',
    email_masked   STRING    COMMENT 'PII safe: ****@domain.com',
    city           STRING    COMMENT 'Uppercased, trimmed city name',
    signup_date    DATE      COMMENT 'Parsed signup date',
    received_date  DATE      COMMENT 'Bronze received_date (lineage)',
    source_file    STRING    COMMENT 'Source CSV path (lineage)',
    ingestion_ts   TIMESTAMP COMMENT 'Bronze ingestion timestamp (lineage)',
    start_date     DATE      COMMENT 'SCD2: when this version became active',
    end_date       DATE      COMMENT 'SCD2: when this version was closed (NULL = still active)',
    is_current     BOOLEAN   COMMENT 'SCD2: TRUE for the one active version per customer_id'
)
USING DELTA
COMMENT 'Silver dim_customer with SCD Type 2 history. Query with is_current=true for current state.'
"""

DDL_SILVER_DIM_PRODUCT_SCD2 = f"""
CREATE TABLE IF NOT EXISTS {PRODUCT_SILVER} (
    product_id     STRING    NOT NULL COMMENT 'Business key from source',
    product_name   STRING    COMMENT 'Cleaned product name',
    category       STRING    COMMENT 'Uppercased, trimmed category',
    price          DOUBLE    COMMENT 'Parsed price — NULL for 357 products with missing price data',
    received_date  DATE      COMMENT 'Bronze received_date (lineage)',
    source_file    STRING    COMMENT 'Source CSV path (lineage)',
    ingestion_ts   TIMESTAMP COMMENT 'Bronze ingestion timestamp (lineage)',
    start_date     DATE      COMMENT 'SCD2: when this version became active',
    end_date       DATE      COMMENT 'SCD2: when this version was closed (NULL = still active)',
    is_current     BOOLEAN   COMMENT 'SCD2: TRUE for the one active version per product_id'
)
USING DELTA
COMMENT 'Silver dim_product with SCD Type 2 history. Products with NULL price cause sales quarantine.'
"""

DDL_SILVER_FACT_SALES_CLEAN = f"""
CREATE TABLE IF NOT EXISTS {SALES_CLEAN} (
    order_id      STRING    NOT NULL COMMENT 'Business key — unique per transaction',
    customer_id   STRING    COMMENT 'FK to dim_customer_scd2',
    product_id    STRING    COMMENT 'FK to dim_product_scd2',
    quantity      INT       COMMENT 'Parsed order quantity',
    price         DOUBLE    COMMENT 'Unit price at time of sale — guaranteed NOT NULL',
    order_date    DATE      COMMENT 'Parsed order date',
    total_amount  DOUBLE    COMMENT 'quantity * price — guaranteed NOT NULL. Used in dashboards.',
    received_date DATE      COMMENT 'Bronze received_date (lineage)',
    source_file   STRING    COMMENT 'Source CSV path (lineage)',
    ingestion_ts  TIMESTAMP COMMENT 'Bronze ingestion timestamp (lineage)'
)
USING DELTA
PARTITIONED BY (order_date)
COMMENT 'Silver clean sales. ONLY rows where price resolved. No NULLs in total_amount. Gold source.'
"""

DDL_SILVER_FACT_SALES_QUARANTINE = f"""
CREATE TABLE IF NOT EXISTS {SALES_QUARANTINE} (
    order_id          STRING    NOT NULL COMMENT 'Business key',
    customer_id       STRING    COMMENT 'FK to customers',
    product_id        STRING    COMMENT 'The unresolvable product_id',
    quantity          INT       COMMENT 'Parsed quantity',
    price             DOUBLE    COMMENT 'NULL — reason this row is quarantined',
    order_date        DATE      COMMENT 'Parsed order date',
    total_amount      DOUBLE    COMMENT 'NULL — cannot compute without price',
    quarantine_reason STRING    COMMENT 'Why: PRODUCT_PRICE_IS_NULL or PRODUCT_NOT_IN_DIM',
    quarantine_ts     TIMESTAMP COMMENT 'When this row entered quarantine',
    resolution_status STRING    COMMENT 'PENDING (not fixed) or RESOLVED (price found, promoted)',
    received_date     DATE      COMMENT 'Bronze received_date (lineage)',
    source_file       STRING    COMMENT 'Source CSV path (lineage)',
    ingestion_ts      TIMESTAMP COMMENT 'Bronze ingestion timestamp (lineage)'
)
USING DELTA
PARTITIONED BY (resolution_status)
COMMENT 'Quarantine holds sales rows that cannot compute total_amount. PENDING=awaiting fix, RESOLVED=promoted to clean.'
"""


# ══════════════════════════════════════════════════════════════════════════════
# DDL — GOLD TABLES (Star Schema)
# Surrogate keys auto-generated by Delta IDENTITY columns.
# Built exclusively from fact_sales_clean — quarantine rows never reach gold.
# ══════════════════════════════════════════════════════════════════════════════

DDL_GOLD_DIM_CUSTOMER = f"""
CREATE TABLE IF NOT EXISTS {DIM_CUSTOMER} (
    customer_sk   BIGINT    GENERATED ALWAYS AS IDENTITY COMMENT 'Surrogate key (auto-generated)',
    customer_id   STRING    COMMENT 'Business key from source (FK to silver)',
    name          STRING    COMMENT 'Customer full name',
    city          STRING    COMMENT 'Customer city'
)
USING DELTA
COMMENT 'Gold dim_customer — current snapshot only (is_current=true from silver). Use customer_sk in fact_sales.'
"""

DDL_GOLD_DIM_PRODUCT = f"""
CREATE TABLE IF NOT EXISTS {DIM_PRODUCT} (
    product_sk    BIGINT    GENERATED ALWAYS AS IDENTITY COMMENT 'Surrogate key (auto-generated)',
    product_id    STRING    COMMENT 'Business key from source',
    product_name  STRING    COMMENT 'Product display name',
    category      STRING    COMMENT 'Product category',
    price         DOUBLE    COMMENT 'Current unit price'
)
USING DELTA
COMMENT 'Gold dim_product — current snapshot only. Only products with price. Use product_sk in fact_sales.'
"""

DDL_GOLD_DIM_DATE = f"""
CREATE TABLE IF NOT EXISTS {DIM_DATE} (
    date_key     INT     NOT NULL COMMENT 'Date surrogate key in YYYYMMDD format (e.g. 20240115)',
    full_date    DATE    COMMENT 'Actual calendar date',
    year         INT     COMMENT 'Calendar year',
    quarter      INT     COMMENT 'Quarter number (1-4)',
    month        INT     COMMENT 'Month number (1-12)',
    month_name   STRING  COMMENT 'Month name (January, February...)',
    day          INT     COMMENT 'Day of month (1-31)',
    day_of_week  INT     COMMENT 'Day of week (1=Sunday, 7=Saturday in Spark)',
    day_name     STRING  COMMENT 'Day name (Monday, Tuesday...)',
    is_weekend   BOOLEAN COMMENT 'True for Saturday and Sunday'
)
USING DELTA
COMMENT 'Gold dim_date — date spine built from all distinct order_dates in fact_sales_clean.'
"""

DDL_GOLD_FACT_SALES = f"""
CREATE TABLE IF NOT EXISTS {FACT_SALES} (
    order_id      STRING  COMMENT 'Business key — natural key from source',
    customer_sk   BIGINT  COMMENT 'FK to dim_customer.customer_sk',
    product_sk    BIGINT  COMMENT 'FK to dim_product.product_sk',
    date_key      INT     COMMENT 'FK to dim_date.date_key (YYYYMMDD)',
    quantity      INT     COMMENT 'Order quantity',
    unit_price    DOUBLE  COMMENT 'Price per unit at time of sale',
    total_amount  DOUBLE  COMMENT 'quantity * unit_price — primary revenue measure'
)
USING DELTA
PARTITIONED BY (date_key)
COMMENT 'Gold fact_sales — star schema centre. Join to dims via *_sk keys. Partitioned by date_key for BI performance.'
"""


# ══════════════════════════════════════════════════════════════════════════════
# DDL — METADATA TABLES
# ══════════════════════════════════════════════════════════════════════════════

DDL_PROCESSED_FILES = f"""
CREATE TABLE IF NOT EXISTS {META_TABLE} (
    file_name            STRING    COMMENT 'Full ADLS/Volume path of ingested file',
    table_name           STRING    COMMENT 'Target bronze table name (customers/products/sales)',
    processed_timestamp  TIMESTAMP COMMENT 'When this file was recorded as processed',
    row_count            BIGINT    COMMENT 'Number of rows ingested from this file'
)
USING DELTA
COMMENT 'CDC state store. On each run: new_files = all_files_in_folder - files_in_this_table.'
"""

DDL_PIPELINE_RUNS = f"""
CREATE TABLE IF NOT EXISTS {PIPELINE_RUNS} (
    run_id          STRING    COMMENT 'Unique identifier: notebook_rundate_HHMMSS',
    notebook_name   STRING    COMMENT 'Which notebook generated this record',
    run_date        STRING    COMMENT 'Logical run date YYYY-MM-DD',
    start_ts        TIMESTAMP COMMENT 'Run start time',
    end_ts          TIMESTAMP COMMENT 'Run end time',
    status          STRING    COMMENT 'RUNNING | SUCCESS | FAILED',
    rows_processed  BIGINT    COMMENT 'Total rows processed in this run',
    message         STRING    COMMENT 'Summary message or first 500 chars of error'
)
USING DELTA
COMMENT 'Pipeline execution log. Use for SLA monitoring, anomaly detection, and incident investigation.'
"""

DDL_SCHEMA_VERSIONS = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA_VERSIONS} (
    detected_at     TIMESTAMP COMMENT 'When the schema change was detected',
    table_name      STRING    COMMENT 'Which bronze table was affected',
    missing_columns STRING    COMMENT 'Comma-separated list of columns missing from source file',
    extra_columns   STRING    COMMENT 'Comma-separated list of unexpected extra columns',
    source_file     STRING    COMMENT 'The specific file that triggered the drift',
    action_taken    STRING    COMMENT 'NULL_FILLED | KEPT_EXTRA | NO_CHANGE',
    acknowledged    BOOLEAN   COMMENT 'False until data steward reviews and marks as known'
)
USING DELTA
COMMENT 'Schema evolution audit trail. Query WHERE acknowledged=false to see unreviewed drift events.'
"""

DDL_DQ_RESULTS = f"""
CREATE TABLE IF NOT EXISTS {DQ_RESULTS} (
    run_timestamp  TIMESTAMP COMMENT 'When this DQ check ran',
    layer          STRING    COMMENT 'Which layer was checked: bronze/silver/gold',
    table_name     STRING    COMMENT 'Which table was checked',
    check_name     STRING    COMMENT 'Short check identifier (e.g. null_pk_order_id)',
    status         STRING    COMMENT 'PASS or FAIL',
    failed_count   BIGINT    COMMENT 'Number of rows that failed this check (0 if PASS)',
    details        STRING    COMMENT 'Human-readable description of result'
)
USING DELTA
COMMENT 'Data quality results. Use to track DQ trends over time and alert on regressions.'
"""

DDL_QUARANTINE_LOG = f"""
CREATE TABLE IF NOT EXISTS {QUARANTINE_LOG} (
    run_timestamp       TIMESTAMP COMMENT 'When this log entry was written',
    entity              STRING    COMMENT 'Which dataset: sales / customers / products',
    quarantine_reason   STRING    COMMENT 'Reason code for this batch of quarantined rows',
    rows_quarantined    BIGINT    COMMENT 'How many rows went to quarantine this run',
    rows_clean          BIGINT    COMMENT 'How many rows went to clean table this run',
    quarantine_rate_pct DOUBLE    COMMENT 'rows_quarantined / total * 100'
)
USING DELTA
COMMENT 'Quarantine trend log. Rising quarantine_rate_pct week-over-week = source system degrading.'
"""


# ══════════════════════════════════════════════════════════════════════════════
# EXECUTION — CREATE EVERYTHING
# ══════════════════════════════════════════════════════════════════════════════

def run_setup():
    print("╔══════════════════════════════════════════════════════════╗")
    print("║  RETAIL DATA PLATFORM — ONE-TIME SETUP                  ║")
    print("╚══════════════════════════════════════════════════════════╝")

    # ── Catalog ───────────────────────────────────────────────────────────────
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    print(f"\n✓ Catalog: {CATALOG}")

    # ── Schemas ───────────────────────────────────────────────────────────────
    for schema_name in ["01_bronze", "02_silver", "03_gold", "04_metadata"]:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}")
        print(f"✓ Schema : {CATALOG}.{schema_name}")

    # ── Bronze tables ─────────────────────────────────────────────────────────
    print("\n── Bronze ──────────────────────────────────────────────────")
    for ddl, name in [
        (DDL_BRONZE_CUSTOMERS, CUSTOMER_TABLE),
        (DDL_BRONZE_PRODUCTS,  PRODUCT_TABLE),
        (DDL_BRONZE_SALES,     SALES_TABLE),
    ]:
        spark.sql(ddl)
        print(f"  ✓ {name}")

    # ── Silver tables ─────────────────────────────────────────────────────────
    print("\n── Silver ──────────────────────────────────────────────────")
    for ddl, name in [
        (DDL_SILVER_DIM_CUSTOMER_SCD2,      CUSTOMER_SILVER),
        (DDL_SILVER_DIM_PRODUCT_SCD2,       PRODUCT_SILVER),
        (DDL_SILVER_FACT_SALES_CLEAN,       SALES_CLEAN),
        (DDL_SILVER_FACT_SALES_QUARANTINE,  SALES_QUARANTINE),
    ]:
        spark.sql(ddl)
        print(f"  ✓ {name}")

    # ── Gold tables ───────────────────────────────────────────────────────────
    print("\n── Gold ────────────────────────────────────────────────────")
    for ddl, name in [
        (DDL_GOLD_DIM_CUSTOMER, DIM_CUSTOMER),
        (DDL_GOLD_DIM_PRODUCT,  DIM_PRODUCT),
        (DDL_GOLD_DIM_DATE,     DIM_DATE),
        (DDL_GOLD_FACT_SALES,   FACT_SALES),
    ]:
        spark.sql(ddl)
        print(f"  ✓ {name}")

    # ── Metadata tables ───────────────────────────────────────────────────────
    print("\n── Metadata ────────────────────────────────────────────────")
    for ddl, name in [
        (DDL_PROCESSED_FILES,  META_TABLE),
        (DDL_PIPELINE_RUNS,    PIPELINE_RUNS),
        (DDL_SCHEMA_VERSIONS,  SCHEMA_VERSIONS),
        (DDL_DQ_RESULTS,       DQ_RESULTS),
        (DDL_QUARANTINE_LOG,   QUARANTINE_LOG),
    ]:
        spark.sql(ddl)
        print(f"  ✓ {name}")

    print("""
╔══════════════════════════════════════════════════════════════╗
║  SETUP COMPLETE                                              ║
╠══════════════════════════════════════════════════════════════╣
║  Bronze  : customers, products, sales                        ║
║  Silver  : dim_customer_scd2, dim_product_scd2               ║
║            fact_sales_clean    ← price resolved rows         ║
║            fact_sales_quarantine ← pending / resolved rows   ║
║  Gold    : dim_customer, dim_product, dim_date, fact_sales   ║
║  Metadata: processed_files, pipeline_runs, schema_versions   ║
║            dq_results, quarantine_log                        ║
╚══════════════════════════════════════════════════════════════╝
    """)

run_setup()

In [0]:
%sql
DROP TABLE retail_data_project.04_metadata.dq_results;

In [0]:
%sql
CREATE TABLE retail_data_project.04_metadata.dq_results (
  run_timestamp TIMESTAMP,
  notebook STRING,
  layer STRING,
  table_name STRING,
  check_id STRING,
  description STRING,
  status STRING,
  failed_count BIGINT,
  total_count BIGINT,
  fail_rate_pct FLOAT,
  details STRING
);